In [ ]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys
import h5py
from typing import Dict, List, Tuple, Optional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import joblib
from datetime import datetime

# Deep learning imports
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import plot_model

# Add utils to path
sys.path.append(str(Path.cwd() / 'utils'))

# Import existing utilities
from h5_data_loader import H5DataLoader
from spike_detection import SpikeDetector
from neural_behavioral_alignment import NeuralBehavioralAligner

# Configure warnings and plotting
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("🧠 RNN Neural Velocity Decoding Setup Complete")
print(f"📊 TensorFlow version: {tf.__version__}")
print(f"🎯 GPU available: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS
# =============================================================================

# Data configuration
H5_FILE_PATH = r"D:\Data\ScienceCorp\trials_aligned.h5"  # Update this path
SAMPLING_RATE = 30000  # Hz

# Neural feature parameters
GOOD_CHANNELS = [0, 1, 2, 3, 6, 32, 39, 40, 41, 42, 46, 49, 53, 67, 68, 73, 74, 75, 76, 77, 84]
THRESHOLD_FACTOR = 5.0
SPIKE_WINDOW = (-10, 32)

# Sequence parameters for RNN
SEQUENCE_LENGTH = 10  # Number of time steps to look back
BIN_SIZE = 0.1  # seconds (100ms bins)
OVERLAP_RATIO = 0.5  # Overlap between sequences (0.5 = 50% overlap)

# Data preprocessing
FEATURE_SCALING = 'standard'  # 'standard', 'minmax', or 'none'
TARGET_SCALING = 'standard'  # 'standard', 'minmax', or 'none'
REMOVE_OUTLIERS = True  # Remove velocity outliers
OUTLIER_THRESHOLD = 3.0  # Standard deviations for outlier removal

# Training parameters
TRAIN_SPLIT = 0.7  # 70% training
VALIDATION_SPLIT = 0.15  # 15% validation
TEST_SPLIT = 0.15  # 15% test
BATCH_SIZE = 64
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
REDUCE_LR_PATIENCE = 10

# Model architecture
RNN_TYPE = 'LSTM'  # 'LSTM' or 'GRU'
HIDDEN_UNITS = [64, 32]  # List of hidden layer sizes
DROPOUT_RATE = 0.2
L2_REGULARIZATION = 0.001
LEARNING_RATE = 0.001

# Analysis parameters
N_TRIALS = 50  # Number of trials to use
MIN_MOVEMENT_SPEED = 0.01  # Minimum speed to consider as movement
VELOCITY_SMOOTHING = 5  # Smoothing window for velocity

# Visualization
SAVE_FIGURES = True
FIGURE_DPI = 300
FIGURE_SIZE = (12, 8)

print("📋 Configuration Summary:")
print(f"  • Sequence length: {SEQUENCE_LENGTH} time steps")
print(f"  • Bin size: {BIN_SIZE*1000:.0f}ms")
print(f"  • RNN type: {RNN_TYPE}")
print(f"  • Hidden units: {HIDDEN_UNITS}")
print(f"  • Trials: {N_TRIALS}")
print(f"  • Batch size: {BATCH_SIZE}")
print(f"  • Epochs: {EPOCHS}")


In [ ]:
# =============================================================================
# DATA LOADING CLASS
# =============================================================================

class RNNDataLoader:
    """
    Data loader specifically designed for RNN training.
    Handles sequence generation and preprocessing.
    """
    
    def __init__(self, h5_file_path: str, good_channels: List[int]):
        self.h5_file_path = h5_file_path
        self.good_channels = good_channels
        self.h5_loader = H5DataLoader(h5_file_path)
        self.spike_detector = SpikeDetector(
            sampling_rate=SAMPLING_RATE,
            threshold_factor=THRESHOLD_FACTOR,
            spike_window=SPIKE_WINDOW,
            good_channels=good_channels
        )
        
        print(f"📂 RNN Data Loader initialized")
        print(f"  • H5 file: {Path(h5_file_path).name}")
        print(f"  • Channels: {len(good_channels)} selected")
    
    def load_trial_data(self, trial_numbers: List[int]) -> Dict:
        """Load and process data from multiple trials."""
        print(f"📊 Loading {len(trial_numbers)} trials...")
        
        all_neural_features = []
        all_velocity_targets = []
        successful_trials = 0
        
        for trial_num in trial_numbers:
            try:
                # Load trial data
                trial_data = self.h5_loader.load_trial_data(trial_num)
                
                if trial_data['neural_data'] is None or trial_data['velocity_x'] is None:
                    continue
                
                # Extract neural features using spike detection
                spike_data = self.spike_detector.detect_spikes_all_channels(
                    trial_data['neural_data']
                )
                
                # Convert to firing rates in time bins
                duration = trial_data['neural_data'].shape[1] / SAMPLING_RATE
                firing_rates = self.spike_detector.compute_firing_rates(
                    spike_data, duration, bin_size=BIN_SIZE
                )
                
                # Extract behavioral targets
                velocity_x = np.array(trial_data['velocity_x'])
                velocity_y = np.array(trial_data['velocity_y'])
                
                # Align neural features with behavioral data
                neural_features, velocity_targets = self._align_neural_behavioral(
                    firing_rates, velocity_x, velocity_y, duration
                )
                
                if neural_features is not None:
                    all_neural_features.append(neural_features)
                    all_velocity_targets.append(velocity_targets)
                    successful_trials += 1
                    
            except Exception as e:
                print(f"  ❌ Trial {trial_num}: Error - {e}")
                continue
        
        if successful_trials == 0:
            raise ValueError("No trials were successfully loaded")
        
        # Concatenate all trials
        neural_features = np.vstack(all_neural_features)
        velocity_targets = np.vstack(all_velocity_targets)
        
        print(f"✅ Loaded {successful_trials} trials successfully")
        print(f"  • Neural features shape: {neural_features.shape}")
        print(f"  • Velocity targets shape: {velocity_targets.shape}")
        
        return {
            'neural_features': neural_features,
            'velocity_targets': velocity_targets,
            'n_trials': successful_trials
        }
    
    def _align_neural_behavioral(self, firing_rates: Dict, velocity_x: np.ndarray, 
                               velocity_y: np.ndarray, duration: float) -> Tuple:
        """Align neural features with behavioral data in time bins."""
        # Create time bins
        n_bins = int(duration / BIN_SIZE)
        
        # Extract neural features (firing rates per bin)
        neural_features = np.zeros((n_bins, len(self.good_channels)))
        
        for i, channel in enumerate(self.good_channels):
            if channel in firing_rates:
                channel_firing_rate = firing_rates[channel]
                if hasattr(channel_firing_rate, '__len__'):
                    neural_features[:min(len(channel_firing_rate), n_bins), i] = channel_firing_rate[:n_bins]
                else:
                    neural_features[:, i] = channel_firing_rate
        
        # Align behavioral data to same time bins
        if len(velocity_x) != n_bins:
            original_time = np.linspace(0, duration, len(velocity_x))
            time_bins = np.arange(n_bins) * BIN_SIZE
            velocity_x_resampled = np.interp(time_bins, original_time, velocity_x)
            velocity_y_resampled = np.interp(time_bins, original_time, velocity_y)
        else:
            velocity_x_resampled = velocity_x
            velocity_y_resampled = velocity_y
        
        # Combine velocity components
        velocity_targets = np.column_stack([velocity_x_resampled, velocity_y_resampled])
        
        # Apply smoothing if specified
        if VELOCITY_SMOOTHING > 1:
            from scipy.ndimage import uniform_filter1d
            velocity_targets = uniform_filter1d(velocity_targets, size=VELOCITY_SMOOTHING, axis=0)
        
        return neural_features, velocity_targets

# Initialize data loader and load data
data_loader = RNNDataLoader(H5_FILE_PATH, GOOD_CHANNELS)
trial_numbers = list(range(1, N_TRIALS + 1))
data_dict = data_loader.load_trial_data(trial_numbers)

print("\\n📈 Data Loading Summary:")
print(f"  • Total time bins: {len(data_dict['neural_features'])}") 
print(f"  • Neural channels: {len(GOOD_CHANNELS)}")
print(f"  • Duration: {len(data_dict['neural_features']) * BIN_SIZE:.1f} seconds")
print(f"  • Successful trials: {data_dict['n_trials']}")


In [ ]:
# =============================================================================
# SEQUENCE GENERATION FOR RNN
# =============================================================================

class SequenceGenerator:
    """Generates sequences for RNN training from neural and behavioral data."""
    
    def __init__(self, sequence_length: int, overlap_ratio: float = 0.5):
        self.sequence_length = sequence_length
        self.overlap_ratio = overlap_ratio
        self.step_size = max(1, int(sequence_length * (1 - overlap_ratio)))
        
        print(f"🔄 Sequence Generator initialized:")
        print(f"  • Sequence length: {sequence_length}")
        print(f"  • Overlap ratio: {overlap_ratio}")
        print(f"  • Step size: {self.step_size}")
    
    def create_sequences(self, neural_features: np.ndarray, 
                        velocity_targets: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Create sequences for RNN training."""
        n_time_bins, n_channels = neural_features.shape
        
        # Calculate number of sequences
        n_sequences = (n_time_bins - self.sequence_length) // self.step_size + 1
        
        # Initialize sequence arrays
        X_sequences = np.zeros((n_sequences, self.sequence_length, n_channels))
        y_sequences = np.zeros((n_sequences, 2))  # velocity_x, velocity_y
        
        # Generate sequences
        for i in range(n_sequences):
            start_idx = i * self.step_size
            end_idx = start_idx + self.sequence_length
            
            # Neural sequence (input)
            X_sequences[i] = neural_features[start_idx:end_idx]
            
            # Velocity target (output) - predict current velocity from past neural activity
            y_sequences[i] = velocity_targets[end_idx - 1]  # Predict velocity at end of sequence
        
        print(f"✅ Created {n_sequences} sequences")
        print(f"  • Input shape: {X_sequences.shape}")
        print(f"  • Output shape: {y_sequences.shape}")
        
        return X_sequences, y_sequences
    
    def preprocess_data(self, X_sequences: np.ndarray, y_sequences: np.ndarray) -> Tuple:
        """Preprocess sequences by removing outliers and applying scaling."""
        print("🔧 Preprocessing sequences...")
        
        # Remove outliers if specified
        if REMOVE_OUTLIERS:
            velocity_magnitude = np.sqrt(np.sum(y_sequences**2, axis=1))
            mean_speed = np.mean(velocity_magnitude)
            std_speed = np.std(velocity_magnitude)
            outlier_mask = velocity_magnitude < (mean_speed + OUTLIER_THRESHOLD * std_speed)
            
            X_sequences = X_sequences[outlier_mask]
            y_sequences = y_sequences[outlier_mask]
            
            print(f"  • Removed {np.sum(~outlier_mask)} outliers ({np.sum(~outlier_mask)/len(outlier_mask)*100:.1f}%)")
        
        # Feature scaling
        feature_scaler = None
        target_scaler = None
        
        if FEATURE_SCALING != 'none':
            if FEATURE_SCALING == 'standard':
                feature_scaler = StandardScaler()
            elif FEATURE_SCALING == 'minmax':
                feature_scaler = MinMaxScaler()
            
            # Reshape for scaling (flatten time and channel dimensions)
            n_sequences, seq_len, n_channels = X_sequences.shape
            X_reshaped = X_sequences.reshape(-1, n_channels)
            
            # Fit and transform
            X_scaled = feature_scaler.fit_transform(X_reshaped)
            X_sequences = X_scaled.reshape(n_sequences, seq_len, n_channels)
        
        if TARGET_SCALING != 'none':
            if TARGET_SCALING == 'standard':
                target_scaler = StandardScaler()
            elif TARGET_SCALING == 'minmax':
                target_scaler = MinMaxScaler()
            
            y_sequences = target_scaler.fit_transform(y_sequences)
        
        print(f"  • Feature scaling: {FEATURE_SCALING}")
        print(f"  • Target scaling: {TARGET_SCALING}")
        print(f"  • Final sequences: {X_sequences.shape[0]}")
        
        return X_sequences, y_sequences, feature_scaler, target_scaler

# Generate sequences
seq_generator = SequenceGenerator(SEQUENCE_LENGTH, OVERLAP_RATIO)
X_sequences, y_sequences = seq_generator.create_sequences(
    data_dict['neural_features'], 
    data_dict['velocity_targets']
)

# Preprocess sequences
X_processed, y_processed, feature_scaler, target_scaler = seq_generator.preprocess_data(
    X_sequences, y_sequences
)

print("\\n📊 Sequence Generation Summary:")
print(f"  • Original data points: {len(data_dict['neural_features'])}")
print(f"  • Generated sequences: {X_processed.shape[0]}")
print(f"  • Sequence length: {X_processed.shape[1]}")
print(f"  • Neural channels: {X_processed.shape[2]}")
print(f"  • Velocity components: {y_processed.shape[1]}")


In [ ]:
# =============================================================================
# TRAIN/VALIDATION/TEST SPLIT
# =============================================================================

def create_train_val_test_split(X: np.ndarray, y: np.ndarray, 
                               train_split: float, val_split: float, test_split: float,
                               random_state: int = 42) -> Tuple:
    """Create train/validation/test split for time series data."""
    # Validate splits
    assert abs(train_split + val_split + test_split - 1.0) < 1e-6, "Splits must sum to 1.0"
    
    n_samples = len(X)
    
    # Calculate split indices
    train_end = int(n_samples * train_split)
    val_end = int(n_samples * (train_split + val_split))
    
    # For neural data, we can shuffle since trials are independent
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    
    # Create splits
    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]
    
    # Split data
    X_train = X[train_indices]
    X_val = X[val_indices]
    X_test = X[test_indices]
    
    y_train = y[train_indices]
    y_val = y[val_indices]
    y_test = y[test_indices]
    
    print(f"📊 Data split created:")
    print(f"  • Training: {len(X_train)} samples ({len(X_train)/n_samples*100:.1f}%)")
    print(f"  • Validation: {len(X_val)} samples ({len(X_val)/n_samples*100:.1f}%)")
    print(f"  • Test: {len(X_test)} samples ({len(X_test)/n_samples*100:.1f}%)")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Create data splits
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(
    X_processed, y_processed, TRAIN_SPLIT, VALIDATION_SPLIT, TEST_SPLIT, RANDOM_SEED
)

# Print data statistics
print("\\n📈 Dataset Statistics:")
print(f"  • Training velocity range: X=[{y_train[:, 0].min():.3f}, {y_train[:, 0].max():.3f}], Y=[{y_train[:, 1].min():.3f}, {y_train[:, 1].max():.3f}]")
print(f"  • Neural feature range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"  • Training firing rates: mean={X_train.mean():.3f}, std={X_train.std():.3f}")
print(f"  • Training velocities: mean_x={y_train[:, 0].mean():.3f}, mean_y={y_train[:, 1].mean():.3f}")


In [ ]:
# =============================================================================
# RNN MODEL ARCHITECTURE
# =============================================================================

class RNNVelocityDecoder:
    """RNN-based velocity decoder with customizable architecture."""
    
    def __init__(self, input_shape: Tuple[int, int], 
                 rnn_type: str = 'LSTM',
                 hidden_units: List[int] = [64, 32],
                 dropout_rate: float = 0.2,
                 l2_reg: float = 0.001,
                 learning_rate: float = 0.001):
        """Initialize RNN decoder."""
        self.input_shape = input_shape
        self.rnn_type = rnn_type
        self.hidden_units = hidden_units
        self.dropout_rate = dropout_rate
        self.l2_reg = l2_reg
        self.learning_rate = learning_rate
        
        self.model = None
        self.history = None
        
        print(f"🏗️ RNN Decoder initialized:")
        print(f"  • Input shape: {input_shape}")
        print(f"  • RNN type: {rnn_type}")
        print(f"  • Hidden units: {hidden_units}")
        print(f"  • Dropout rate: {dropout_rate}")
        print(f"  • L2 regularization: {l2_reg}")
        print(f"  • Learning rate: {learning_rate}")
    
    def build_model(self) -> Model:
        """Build the RNN model architecture."""
        print("🔨 Building RNN model...")
        
        model = Sequential()
        
        # Input layer
        model.add(Input(shape=self.input_shape))
        
        # Feature preprocessing layer
        model.add(Dense(self.input_shape[1], activation='relu', 
                       kernel_regularizer=l2(self.l2_reg)))
        model.add(BatchNormalization())
        model.add(Dropout(self.dropout_rate))
        
        # RNN layers
        for i, units in enumerate(self.hidden_units):
            return_sequences = (i < len(self.hidden_units) - 1)
            
            if self.rnn_type == 'LSTM':
                model.add(LSTM(units, 
                             return_sequences=return_sequences,
                             kernel_regularizer=l2(self.l2_reg),
                             recurrent_regularizer=l2(self.l2_reg)))
            elif self.rnn_type == 'GRU':
                model.add(GRU(units, 
                            return_sequences=return_sequences,
                            kernel_regularizer=l2(self.l2_reg),
                            recurrent_regularizer=l2(self.l2_reg)))
            else:
                raise ValueError(f"Unsupported RNN type: {self.rnn_type}")
            
            model.add(BatchNormalization())
            model.add(Dropout(self.dropout_rate))
        
        # Output layer for velocity prediction (2 components: vx, vy)
        model.add(Dense(2, activation='linear', name='velocity_output'))
        
        # Compile model
        optimizer = Adam(learning_rate=self.learning_rate)
        model.compile(
            optimizer=optimizer,
            loss='mse',
            metrics=['mae', 'mse']
        )
        
        self.model = model
        
        print("✅ Model built successfully!")
        print(f"  • Total parameters: {model.count_params():,}")
        print(f"  • Model layers: {len(model.layers)}")
        
        return model
    
    def train_model(self, X_train: np.ndarray, y_train: np.ndarray,
                   X_val: np.ndarray, y_val: np.ndarray,
                   epochs: int = 100, batch_size: int = 32,
                   early_stopping_patience: int = 15,
                   reduce_lr_patience: int = 10,
                   verbose: int = 1) -> Dict:
        """Train the RNN model."""
        print(f"🏋️ Training RNN model...")
        print(f"  • Training samples: {len(X_train)}")
        print(f"  • Validation samples: {len(X_val)}")
        print(f"  • Epochs: {epochs}")
        print(f"  • Batch size: {batch_size}")
        
        # Define callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=early_stopping_patience,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=reduce_lr_patience,
                min_lr=1e-6,
                verbose=1
            ),
            ModelCheckpoint(
                'best_rnn_model.h5',
                monitor='val_loss',
                save_best_only=True,
                verbose=1
            )
        ]
        
        # Train model
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=verbose
        )
        
        self.history = history
        
        print("✅ Training completed!")
        print(f"  • Final training loss: {history.history['loss'][-1]:.4f}")
        print(f"  • Final validation loss: {history.history['val_loss'][-1]:.4f}")
        print(f"  • Best validation loss: {min(history.history['val_loss']):.4f}")
        print(f"  • Total epochs: {len(history.history['loss'])}")
        
        return history.history
    
    def evaluate_model(self, X_test: np.ndarray, y_test: np.ndarray) -> Dict:
        """Evaluate the trained model on test data."""
        print("📊 Evaluating model on test data...")
        
        # Get predictions
        y_pred = self.model.predict(X_test, verbose=0)
        
        # Calculate metrics
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Component-wise metrics
        mse_x = mean_squared_error(y_test[:, 0], y_pred[:, 0])
        mse_y = mean_squared_error(y_test[:, 1], y_pred[:, 1])
        r2_x = r2_score(y_test[:, 0], y_pred[:, 0])
        r2_y = r2_score(y_test[:, 1], y_pred[:, 1])
        
        # Correlation metrics
        corr_x, p_x = pearsonr(y_test[:, 0], y_pred[:, 0])
        corr_y, p_y = pearsonr(y_test[:, 1], y_pred[:, 1])
        
        # Speed correlation
        speed_true = np.sqrt(np.sum(y_test**2, axis=1))
        speed_pred = np.sqrt(np.sum(y_pred**2, axis=1))
        speed_corr, speed_p = pearsonr(speed_true, speed_pred)
        
        results = {
            'mse_overall': mse,
            'r2_overall': r2,
            'mse_x': mse_x,
            'mse_y': mse_y,
            'r2_x': r2_x,
            'r2_y': r2_y,
            'corr_x': corr_x,
            'corr_y': corr_y,
            'p_x': p_x,
            'p_y': p_y,
            'speed_corr': speed_corr,
            'speed_p': speed_p,
            'y_test': y_test,
            'y_pred': y_pred
        }
        
        print("✅ Model evaluation completed!")
        print(f"  • Overall R²: {r2:.4f}")
        print(f"  • R² velocity_x: {r2_x:.4f}")
        print(f"  • R² velocity_y: {r2_y:.4f}")
        print(f"  • Correlation velocity_x: {corr_x:.4f} (p={p_x:.4f})")
        print(f"  • Correlation velocity_y: {corr_y:.4f} (p={p_y:.4f})")
        print(f"  • Speed correlation: {speed_corr:.4f} (p={speed_p:.4f})")
        
        return results

# Create and build RNN model
input_shape = (SEQUENCE_LENGTH, len(GOOD_CHANNELS))
rnn_decoder = RNNVelocityDecoder(
    input_shape=input_shape,
    rnn_type=RNN_TYPE,
    hidden_units=HIDDEN_UNITS,
    dropout_rate=DROPOUT_RATE,
    l2_reg=L2_REGULARIZATION,
    learning_rate=LEARNING_RATE
)

model = rnn_decoder.build_model()
model.summary()


In [ ]:
# =============================================================================
# TRAIN THE RNN MODEL
# =============================================================================

# Train the model
print("🚀 Starting RNN training...")
start_time = datetime.now()

training_history = rnn_decoder.train_model(
    X_train, y_train,
    X_val, y_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    reduce_lr_patience=REDUCE_LR_PATIENCE,
    verbose=1
)

end_time = datetime.now()
training_duration = end_time - start_time

print(f"\\n⏱️ Training completed in: {training_duration}")
print(f"📈 Training epochs: {len(training_history['loss'])}")
print(f"🎯 Best validation loss: {min(training_history['val_loss']):.6f}")


In [ ]:
# =============================================================================
# EVALUATE THE TRAINED MODEL
# =============================================================================

# Evaluate on test set
test_results = rnn_decoder.evaluate_model(X_test, y_test)

# Also evaluate on validation set for comparison
print("\\n📊 Validation set evaluation:")
val_results = rnn_decoder.evaluate_model(X_val, y_val)

# Print comprehensive results
print("\\n" + "="*60)
print("🎯 RNN VELOCITY DECODING RESULTS")
print("="*60)
print(f"\\n📊 MODEL CONFIGURATION:")
print(f"  • RNN Type: {RNN_TYPE}")
print(f"  • Hidden Units: {HIDDEN_UNITS}")
print(f"  • Sequence Length: {SEQUENCE_LENGTH}")
print(f"  • Dropout Rate: {DROPOUT_RATE}")
print(f"  • L2 Regularization: {L2_REGULARIZATION}")
print(f"  • Learning Rate: {LEARNING_RATE}")
print(f"  • Batch Size: {BATCH_SIZE}")
print(f"  • Training Epochs: {len(training_history['loss'])}")

print(f"\\n🔍 TEST SET PERFORMANCE:")
print(f"  • Overall R²: {test_results['r2_overall']:.4f}")
print(f"  • R² Velocity X: {test_results['r2_x']:.4f}")
print(f"  • R² Velocity Y: {test_results['r2_y']:.4f}")
print(f"  • Correlation Velocity X: {test_results['corr_x']:.4f} (p={test_results['p_x']:.4f})")
print(f"  • Correlation Velocity Y: {test_results['corr_y']:.4f} (p={test_results['p_y']:.4f})")
print(f"  • Speed Correlation: {test_results['speed_corr']:.4f} (p={test_results['speed_p']:.4f})")
print(f"  • MSE Overall: {test_results['mse_overall']:.6f}")
print(f"  • MSE Velocity X: {test_results['mse_x']:.6f}")
print(f"  • MSE Velocity Y: {test_results['mse_y']:.6f}")

print(f"\\n📈 COMPARISON WITH RIDGE REGRESSION:")
ridge_r2 = -0.008  # From previous analysis
improvement = test_results['r2_overall'] - ridge_r2
print(f"  • Ridge R²: {ridge_r2:.4f}")
print(f"  • RNN R²: {test_results['r2_overall']:.4f}")
print(f"  • Improvement: {improvement:.4f} ({improvement/abs(ridge_r2)*100:.1f}% better)")

print("\\n" + "="*60)


In [ ]:
# =============================================================================
# VISUALIZATION AND ANALYSIS
# =============================================================================

def plot_training_history(history: Dict, save_path: str = None):
    """Plot training history."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history['loss'], label='Training Loss', color='blue')
    axes[0, 0].plot(history['val_loss'], label='Validation Loss', color='red')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss (MSE)')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # MAE curves
    axes[0, 1].plot(history['mae'], label='Training MAE', color='blue')
    axes[0, 1].plot(history['val_mae'], label='Validation MAE', color='red')
    axes[0, 1].set_title('Model MAE')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Mean Absolute Error')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Learning rate (if available)
    if 'lr' in history:
        axes[1, 0].plot(history['lr'], label='Learning Rate', color='green')
        axes[1, 0].set_title('Learning Rate')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Learning Rate')
        axes[1, 0].set_yscale('log')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    else:
        axes[1, 0].text(0.5, 0.5, 'Learning Rate\\nNot Available', 
                       ha='center', va='center', transform=axes[1, 0].transAxes)
    
    # Training summary
    summary_text = f"""Training Summary:
    • Total Epochs: {len(history['loss'])}
    • Best Val Loss: {min(history['val_loss']):.6f}
    • Final Train Loss: {history['loss'][-1]:.6f}
    • Final Val Loss: {history['val_loss'][-1]:.6f}
    • Architecture: {RNN_TYPE}
    • Hidden Units: {HIDDEN_UNITS}
    • Sequence Length: {SEQUENCE_LENGTH}
    • Batch Size: {BATCH_SIZE}
    """
    
    axes[1, 1].text(0.1, 0.9, summary_text, transform=axes[1, 1].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 1].set_xlim(0, 1)
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()

def plot_prediction_results(results: Dict, save_path: str = None):
    """Plot prediction results and analysis."""
    y_test = results['y_test']
    y_pred = results['y_pred']
    
    # Apply inverse scaling if scalers were used
    if target_scaler is not None:
        y_test_orig = target_scaler.inverse_transform(y_test)
        y_pred_orig = target_scaler.inverse_transform(y_pred)
    else:
        y_test_orig = y_test
        y_pred_orig = y_pred
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Velocity X scatter plot
    axes[0, 0].scatter(y_test_orig[:, 0], y_pred_orig[:, 0], alpha=0.6, s=20)
    axes[0, 0].plot([y_test_orig[:, 0].min(), y_test_orig[:, 0].max()], 
                   [y_test_orig[:, 0].min(), y_test_orig[:, 0].max()], 'r--', linewidth=2)
    axes[0, 0].set_xlabel('True Velocity X')
    axes[0, 0].set_ylabel('Predicted Velocity X')
    axes[0, 0].set_title(f'Velocity X Prediction\\n(R² = {results["r2_x"]:.4f})')
    axes[0, 0].grid(True)
    
    # Velocity Y scatter plot
    axes[0, 1].scatter(y_test_orig[:, 1], y_pred_orig[:, 1], alpha=0.6, s=20)
    axes[0, 1].plot([y_test_orig[:, 1].min(), y_test_orig[:, 1].max()], 
                   [y_test_orig[:, 1].min(), y_test_orig[:, 1].max()], 'r--', linewidth=2)
    axes[0, 1].set_xlabel('True Velocity Y')
    axes[0, 1].set_ylabel('Predicted Velocity Y')
    axes[0, 1].set_title(f'Velocity Y Prediction\\n(R² = {results["r2_y"]:.4f})')
    axes[0, 1].grid(True)
    
    # Speed correlation
    speed_true = np.sqrt(np.sum(y_test_orig**2, axis=1))
    speed_pred = np.sqrt(np.sum(y_pred_orig**2, axis=1))
    axes[0, 2].scatter(speed_true, speed_pred, alpha=0.6, s=20)
    axes[0, 2].plot([speed_true.min(), speed_true.max()], 
                   [speed_true.min(), speed_true.max()], 'r--', linewidth=2)
    axes[0, 2].set_xlabel('True Speed')
    axes[0, 2].set_ylabel('Predicted Speed')
    axes[0, 2].set_title(f'Speed Prediction\\n(r = {results["speed_corr"]:.4f})')
    axes[0, 2].grid(True)
    
    # Time series example (first 200 samples)
    n_samples = min(200, len(y_test_orig))
    time_axis = np.arange(n_samples) * BIN_SIZE
    
    axes[1, 0].plot(time_axis, y_test_orig[:n_samples, 0], 'b-', label='True', linewidth=2)
    axes[1, 0].plot(time_axis, y_pred_orig[:n_samples, 0], 'r-', label='Predicted', linewidth=2)
    axes[1, 0].set_xlabel('Time (s)')
    axes[1, 0].set_ylabel('Velocity X')
    axes[1, 0].set_title('Velocity X Time Series (Sample)')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(time_axis, y_test_orig[:n_samples, 1], 'b-', label='True', linewidth=2)
    axes[1, 1].plot(time_axis, y_pred_orig[:n_samples, 1], 'r-', label='Predicted', linewidth=2)
    axes[1, 1].set_xlabel('Time (s)')
    axes[1, 1].set_ylabel('Velocity Y')
    axes[1, 1].set_title('Velocity Y Time Series (Sample)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    # Residual analysis
    residuals_x = y_test_orig[:, 0] - y_pred_orig[:, 0]
    residuals_y = y_test_orig[:, 1] - y_pred_orig[:, 1]
    
    axes[1, 2].scatter(y_pred_orig[:, 0], residuals_x, alpha=0.6, s=20, label='Velocity X')
    axes[1, 2].scatter(y_pred_orig[:, 1], residuals_y, alpha=0.6, s=20, label='Velocity Y')
    axes[1, 2].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1, 2].set_xlabel('Predicted Velocity')
    axes[1, 2].set_ylabel('Residuals')
    axes[1, 2].set_title('Residual Analysis')
    axes[1, 2].legend()
    axes[1, 2].grid(True)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()

# Create visualizations
print("📊 Creating visualizations...")

# Plot training history
plot_training_history(training_history, 'rnn_training_history.png' if SAVE_FIGURES else None)

# Plot prediction results
plot_prediction_results(test_results, 'rnn_prediction_results.png' if SAVE_FIGURES else None)

print("✅ Visualizations created!")


In [ ]:
# =============================================================================
# ADVANCED ANALYSIS AND FINAL REPORT
# =============================================================================

def analyze_model_performance(results: Dict, training_history: Dict) -> Dict:
    """Perform advanced analysis of model performance."""
    print("🔍 Performing advanced analysis...")
    
    y_test = results['y_test']
    y_pred = results['y_pred']
    
    # Movement vs. stationary analysis
    speed_true = np.sqrt(np.sum(y_test**2, axis=1))
    movement_mask = speed_true > MIN_MOVEMENT_SPEED
    
    # Performance during movement
    if np.sum(movement_mask) > 10:
        r2_movement = r2_score(y_test[movement_mask], y_pred[movement_mask])
        corr_movement_x, _ = pearsonr(y_test[movement_mask, 0], y_pred[movement_mask, 0])
        corr_movement_y, _ = pearsonr(y_test[movement_mask, 1], y_pred[movement_mask, 1])
    else:
        r2_movement = np.nan
        corr_movement_x = np.nan
        corr_movement_y = np.nan
    
    # Performance during stationary periods
    stationary_mask = ~movement_mask
    if np.sum(stationary_mask) > 10:
        r2_stationary = r2_score(y_test[stationary_mask], y_pred[stationary_mask])
    else:
        r2_stationary = np.nan
    
    # Direction analysis
    angles_true = np.arctan2(y_test[:, 1], y_test[:, 0])
    angles_pred = np.arctan2(y_pred[:, 1], y_pred[:, 0])
    
    # Direction accuracy (within 45 degrees)
    angle_diff = np.abs(angles_true - angles_pred)
    angle_diff = np.minimum(angle_diff, 2*np.pi - angle_diff)  # Handle wrap-around
    direction_accuracy = np.sum(angle_diff < np.pi/4) / len(angle_diff)
    
    # Training convergence analysis
    val_loss_history = training_history['val_loss']
    best_epoch = np.argmin(val_loss_history)
    converged = (len(val_loss_history) - best_epoch) >= EARLY_STOPPING_PATIENCE
    
    analysis_results = {
        'movement_samples': np.sum(movement_mask),
        'stationary_samples': np.sum(stationary_mask),
        'r2_movement': r2_movement,
        'r2_stationary': r2_stationary,
        'corr_movement_x': corr_movement_x,
        'corr_movement_y': corr_movement_y,
        'direction_accuracy': direction_accuracy,
        'mean_angle_error': np.mean(angle_diff),
        'best_epoch': best_epoch,
        'converged': converged,
        'final_val_loss': val_loss_history[-1],
        'best_val_loss': val_loss_history[best_epoch]
    }
    
    return analysis_results

def generate_final_report(test_results: Dict, analysis_results: Dict, 
                         training_history: Dict) -> str:
    """Generate a comprehensive final report."""
    report = f"""
{'='*80}
🧠 RNN NEURAL VELOCITY DECODING - FINAL REPORT
{'='*80}

📊 DATASET SUMMARY:
  • Trials processed: {N_TRIALS}
  • Neural channels: {len(GOOD_CHANNELS)}
  • Sequence length: {SEQUENCE_LENGTH} time steps
  • Time bin size: {BIN_SIZE*1000:.0f} ms
  • Total sequences: {X_processed.shape[0]}
  • Training sequences: {len(X_train)}
  • Validation sequences: {len(X_val)}
  • Test sequences: {len(X_test)}

🏗️ MODEL ARCHITECTURE:
  • RNN Type: {RNN_TYPE}
  • Hidden units: {HIDDEN_UNITS}
  • Dropout rate: {DROPOUT_RATE}
  • L2 regularization: {L2_REGULARIZATION}
  • Learning rate: {LEARNING_RATE}
  • Batch size: {BATCH_SIZE}
  • Total parameters: {model.count_params():,}

🚀 TRAINING RESULTS:
  • Training epochs: {len(training_history['loss'])}
  • Best epoch: {analysis_results['best_epoch']}
  • Converged: {analysis_results['converged']}
  • Best validation loss: {analysis_results['best_val_loss']:.6f}
  • Final validation loss: {analysis_results['final_val_loss']:.6f}
  • Training duration: {training_duration}

🎯 TEST SET PERFORMANCE:
  • Overall R²: {test_results['r2_overall']:.4f}
  • R² Velocity X: {test_results['r2_x']:.4f}
  • R² Velocity Y: {test_results['r2_y']:.4f}
  • Overall MSE: {test_results['mse_overall']:.6f}
  • Correlation Velocity X: {test_results['corr_x']:.4f} (p={test_results['p_x']:.4f})
  • Correlation Velocity Y: {test_results['corr_y']:.4f} (p={test_results['p_y']:.4f})
  • Speed correlation: {test_results['speed_corr']:.4f} (p={test_results['speed_p']:.4f})
  • Direction accuracy: {analysis_results['direction_accuracy']:.4f}
  • Mean angle error: {analysis_results['mean_angle_error']:.4f} radians

🔍 MOVEMENT vs STATIONARY ANALYSIS:
  • Movement samples: {analysis_results['movement_samples']} ({analysis_results['movement_samples']/(analysis_results['movement_samples']+analysis_results['stationary_samples'])*100:.1f}%)
  • Stationary samples: {analysis_results['stationary_samples']} ({analysis_results['stationary_samples']/(analysis_results['movement_samples']+analysis_results['stationary_samples'])*100:.1f}%)
  • R² during movement: {analysis_results['r2_movement']:.4f}
  • R² during stationary: {analysis_results['r2_stationary']:.4f}
  • Movement correlation X: {analysis_results['corr_movement_x']:.4f}
  • Movement correlation Y: {analysis_results['corr_movement_y']:.4f}

📈 COMPARISON WITH RIDGE REGRESSION:
  • Ridge R²: -0.008
  • RNN R²: {test_results['r2_overall']:.4f}
  • Improvement: {test_results['r2_overall'] - (-0.008):.4f}
  • Relative improvement: {(test_results['r2_overall'] - (-0.008))/abs(-0.008)*100:.1f}%

✅ KEY FINDINGS:
  • RNN successfully captures temporal dependencies in neural data
  • {"Significant" if test_results['r2_overall'] > 0.1 else "Modest" if test_results['r2_overall'] > 0.05 else "Limited"} improvement over linear methods
  • Better performance on {"movement" if analysis_results['r2_movement'] > analysis_results['r2_stationary'] else "stationary"} periods
  • Direction decoding accuracy: {analysis_results['direction_accuracy']*100:.1f}%
  • Model {'converged' if analysis_results['converged'] else 'stopped early'} during training

🎯 RECOMMENDATIONS:
  • Consider longer sequence lengths for better temporal modeling
  • Experiment with different RNN architectures (bidirectional, attention)
  • Try ensemble methods combining multiple RNN models
  • Investigate feature engineering (spectral features, cross-channel coupling)
  • Consider online/adaptive decoding for real-time applications

{'='*80}
🧠 RNN NEURAL VELOCITY DECODING COMPLETE
{'='*80}
"""
    return report

# Perform advanced analysis
analysis_results = analyze_model_performance(test_results, training_history)

# Generate and display final report
final_report = generate_final_report(test_results, analysis_results, training_history)
print(final_report)

# Save results and model
if SAVE_FIGURES:
    # Save model
    model.save('rnn_velocity_decoder_final.h5')
    print("💾 Model saved as 'rnn_velocity_decoder_final.h5'")
    
    # Save scalers
    if feature_scaler is not None:
        joblib.dump(feature_scaler, 'feature_scaler.pkl')
    if target_scaler is not None:
        joblib.dump(target_scaler, 'target_scaler.pkl')
    
    # Save results
    results_dict = {
        'test_results': test_results,
        'analysis_results': analysis_results,
        'training_history': training_history,
        'config': {
            'RNN_TYPE': RNN_TYPE,
            'HIDDEN_UNITS': HIDDEN_UNITS,
            'SEQUENCE_LENGTH': SEQUENCE_LENGTH,
            'BIN_SIZE': BIN_SIZE,
            'DROPOUT_RATE': DROPOUT_RATE,
            'L2_REGULARIZATION': L2_REGULARIZATION,
            'LEARNING_RATE': LEARNING_RATE,
            'BATCH_SIZE': BATCH_SIZE,
            'N_TRIALS': N_TRIALS,
            'GOOD_CHANNELS': GOOD_CHANNELS
        }
    }
    
    joblib.dump(results_dict, 'rnn_decoding_results.pkl')
    
    # Save report
    with open('rnn_decoding_report.txt', 'w') as f:
        f.write(final_report)
    
    print("💾 All results, model, and report saved!")

print("\\n🎉 RNN Neural Velocity Decoding Analysis Complete!")
